# Fun & Fit 건강 어드바이저 에이전트

**Microsoft Foundry Agent Service**로 재미있으면서도 (주의 문구가 포함된) 건강 & 피트니스 어시스턴트를 만들어봅니다.

이번에 수행할 것:

1. `AIProjectClient`로 프로젝트에 **연결**
2. `PromptAgentDefinition`으로 **에이전트 생성** (전반적인 웰니스·영양 조언 + 주의 문구)
3. **Conversations + Responses API**로 대화 관리
4. 에이전트 **정리(삭제)**

> **변경 참고**: 구버전의 `create_agent` / `threads` / `runs`(Assistants API 기반) 패턴은 2026-08-26 서비스 종료로 최신 패턴으로 교체되었습니다 — [마이그레이션 문서](https://learn.microsoft.com/azure/foundry/agents/how-to/migrate)

## 1. 초기 설정

`AIProjectClient`를 초기화합니다. `az login`이 되어 있어야 합니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv(Path().absolute().parent / ".env")

endpoint = os.environ["PROJECT_ENDPOINT"]
model_name = os.environ["MODEL_NAME"]

project = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("✅ Successfully initialized AIProjectClient")

## 2. Fun & Fit 건강 어드바이저 에이전트 만들기

에이전트의 지시문(instructions)에 **주의 문구**를 명시적으로 포함하여 항상 안전을 우선하도록 합니다. `create_version`은 같은 이름의 에이전트에 새 버전을 만드는 방식이라, 지시문을 수정해 다시 실행하면 버전이 올라갑니다.

In [ ]:
AGENT_NAME = "fun-fit-health-advisor"

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="""
        You are a friendly AI Health Advisor.
        You provide general health, fitness, and nutrition information, but always:
        1. Include medical disclaimers.
        2. Encourage the user to consult healthcare professionals.
        3. Provide general, non-diagnostic advice around wellness, diet, and fitness.
        4. Clearly remind them you're not a doctor.
        5. Encourage safe and balanced approaches to exercise and nutrition.
        """,
    ),
)
print(f"🎉 Created agent '{AGENT_NAME}' (version {agent.version})")

## 3. 대화(Conversation) 시작하기

구버전의 *thread*는 **conversation**으로 대체되었습니다. 에이전트에 바인딩된 OpenAI 클라이언트를 얻고, 건강 Q&A 전용 대화를 만듭니다.

In [ ]:
# 에이전트에 바인딩된 OpenAI 클라이언트
agent_client = project.get_openai_client(agent_name=AGENT_NAME)

# 새 대화 생성 (구 threads.create()에 해당)
conversation = agent_client.conversations.create()
print(f"Created conversation, ID: {conversation.id}")

## 4. 건강 & 피트니스 질문하기

`responses.create()`가 구버전의 message 추가 + run 처리(create_and_process)를 한 번에 대체합니다. 에이전트가 **주의 문구**를 포함해 응답하는지 확인해보세요.

In [ ]:
def ask_health_question(question: str) -> str:
    """대화에 질문을 보내고 에이전트의 응답을 받습니다."""
    response = agent_client.responses.create(
        conversation=conversation.id,   # 대화에 연결 → 이전 문맥 유지
        input=question,
    )
    return response.output_text

q1 = "How do I calculate my BMI, and what does it mean?"
print("🗣️", q1)
print("🤖", ask_health_question(q1))
print()
q2 = "Can you give me a balanced meal plan for someone who exercises 3x a week?"
print("🗣️", q2)
print("🤖", ask_health_question(q2))

### 대화 내역 확인

conversation에 쌓인 아이템(질문·응답)을 나열해 문맥이 유지되는지 확인합니다.

In [ ]:
items = agent_client.conversations.items.list(conversation_id=conversation.id)
for item in items:
    role = getattr(item, "role", item.type)
    text = ""
    for c in getattr(item, "content", []) or []:
        text += getattr(c, "text", "") or ""
    print(f"[{role}] {text[:120]}{'...' if len(text) > 120 else ''}")

## 5. 정리하기 🧹

작업이 완료되면 에이전트와 대화를 삭제합니다. (운영 환경에서는 상태 유지 경험을 위해 에이전트를 계속 유지할 수도 있습니다.)

In [ ]:
# 대화 삭제
agent_client.conversations.delete(conversation.id)
print(f"🗑️ Deleted conversation: {conversation.id}")

# 에이전트 삭제 (모든 버전)
project.agents.delete(agent_name=AGENT_NAME)
print(f"🗑️ Deleted agent: {AGENT_NAME}")

# 축하합니다! 🎉

**Fun & Fit 건강 어드바이저**를 성공적으로 구축했습니다! 이 에이전트는:

1. 기본적인 건강 및 피트니스 질문에 **응답**하고,
2. **주의 문구**로 안전하고 전문적인 상담을 권장하며,
3. **Conversation**으로 문맥을 유지하는 대화를 수행합니다.

포털 **Build > Agents**에서 에이전트가 삭제되었는지 확인하세요. 건강한 코딩 하세요!